## Modeling : phase 1 

In [41]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyClassifier

In [42]:
NOTEBOOKS_DIR=Path.cwd()
ROOT_DIR=NOTEBOOKS_DIR.parent
DIR_DATA_PROCESSED= ROOT_DIR/"data"/"processed"
DIR_DATA_RAW=ROOT_DIR/"data"/"raw"

In [58]:
FEATURE_VERSION="v2"   #change to "v2" to rerun everything on the new features
X_train_features=pd.read_csv(DIR_DATA_PROCESSED / f"x_train_features{'' if FEATURE_VERSION=='v1' else '_v2'}.csv",index_col="ID")

In [59]:
Y_train=pd.read_csv(DIR_DATA_RAW/"y_train.csv",index_col="ID")
print(X_train_features.shape ,Y_train.shape)
print(X_train_features.index.equals(Y_train.index))

(418595, 97) (418595, 1)
True


- We know that lineaire model miss interpret the integer coded columns (should be on hot encoded or used on tree based models that couls handle them) same for categorical features. 
- as a baseline test i decided to drop them 

In [60]:
categorical_cols=["STOCK", "SECTOR", "DATE", "SUB_INDUSTRY", "INDUSTRY", "INDUSTRY_GROUP"]
X=X_train_features.drop(columns=categorical_cols)
y=Y_train["RET"]  
dates=X_train_features["DATE"]  #only to split 

gkf=GroupKFold(n_splits=5)
for fold,(tr,va) in enumerate(gkf.split(X,y,groups=dates)):
    shared=set(dates.iloc[tr]) & set(dates.iloc[va])
    print(f"fold {fold}: train {len(tr)} rows, val {len(va)} rows,shared dates:{len(shared)}")

fold 0: train 335311 rows, val 83284 rows,shared dates:0
fold 1: train 335313 rows, val 83282 rows,shared dates:0
fold 2: train 333127 rows, val 85468 rows,shared dates:0
fold 3: train 335314 rows, val 83281 rows,shared dates:0
fold 4: train 335315 rows, val 83280 rows,shared dates:0


In [61]:
tr,va=next(gkf.split(X,y,groups=dates))
X_tr,X_val=X.iloc[tr],X.iloc[va]
y_tr,y_val=y.iloc[tr],y.iloc[va]
print(X_tr.shape,X_val.shape)

(335311, 91) (83284, 91)


## i) Logistic Reg

- is there any lineaire relationship before using somthing fancy?

In [23]:
model=LogisticRegression(max_iter=1000)
model.fit(X_tr,y_tr)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [24]:
y_hat=model.predict(X_val)
acc=accuracy_score(y_val,y_hat)
print(f"Accuracy of LR : {acc:.4f}")


Accuracy of LR : 0.5007


- with a date based split LR scores 50.7%
- in this validation part always predicting False would score 50.02% so LR is at the baseline: no usable lineaure signal
- below the benchmark 51.31%

In [25]:
print(f"classification report: \n {classification_report(y_val,y_hat)}")

classification report: 
               precision    recall  f1-score   support

       False       0.50      0.48      0.49     41659
        True       0.50      0.52      0.51     41625

    accuracy                           0.50     83284
   macro avg       0.50      0.50      0.50     83284
weighted avg       0.50      0.50      0.50     83284



- precision and recall are between 0.48 and 0.52 for both classes, which is a coin flip level
- the model predicts true slightly ofen recall 0.53 vs 0.48 , small biais
- CC: a lineaire model can't extract signal here we nee tree based models that capture non lineaire interactions 

## ii)  RF

- we use the same model as the benchmark but on our feature eng variables to see if there's any improvment 
- we are getting feature importance for free
- tree based models handles integer coded categories, date is used only to split data, every test date is new so the model cant learn anything reusable from it 

- we gonna use the same hyperparam as the benchmark to have a valide comparison 

In [26]:
X_rf=X_train_features.drop(columns=["DATE"])
X_rf_tr,X_rf_val=X_rf.iloc[tr],X_rf.iloc[va]
y_rf_tr,y_rf_val=y.iloc[tr],y.iloc[va]
print(X_rf_tr.shape,X_rf_val.shape)

(335311, 90) (83284, 90)


In [27]:
rf_model=RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    random_state=42,
    n_jobs=-1 # uses all CPU cores 
)
rf_model.fit(X_rf_tr,y_rf_tr)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total nu

In [28]:
y_hat_rf=rf_model.predict(X_rf_val)
acc_rf=accuracy_score(y_rf_val,y_hat_rf)
print(f"Accuracy of RF  : {acc_rf:.4f}")
print(classification_report(y_rf_val,y_hat_rf))

Accuracy of RF  : 0.5059
              precision    recall  f1-score   support

       False       0.50      0.62      0.56     41659
        True       0.51      0.39      0.44     41625

    accuracy                           0.51     83284
   macro avg       0.51      0.51      0.50     83284
weighted avg       0.51      0.51      0.50     83284



- with date split RF scores 50.59%
- always predicting fals is 50.02% we have 0.5 pointh above that --> a small sign of non lineaire signal
  

In [29]:
importances=pd.Series(rf_model.feature_importances_,index=X_rf_tr.columns)
importances.sort_values(ascending=False).head(20)

RET_std_20      0.050170
VOLUME_1        0.044767
RET_std_5       0.044756
RET_1           0.030416
RET_7           0.027604
RET_REVERSAL    0.027299
RET_mean_20     0.026493
RET_17          0.026465
RET_4           0.024185
RET_3           0.024020
RET_5           0.022973
STOCK           0.022674
RET_14          0.022340
VOLUME_13       0.021628
RET_mean_5      0.021600
RET_2           0.020553
SUB_INDUSTRY    0.019585
RET_10          0.019375
RET_16          0.019279
RET_19          0.018909
dtype: float64

- all 5 feature engineered features are in top 15 
- volume_1 is 2 nd even though we built no volume feature --> ***which motivates volume feature engineering***

In [30]:
X_rf_tr_without_cat,X_rf_val_withoout_cat=X.iloc[tr],X.iloc[va]
y_rf_tr_whithout_cat,y_rf_val_without_cat= y.iloc[tr],y.iloc[va]
print(X_rf_tr_without_cat.shape,X_rf_val_withoout_cat.shape)

(335311, 85) (83284, 85)


In [31]:
rf_model_without_cat=RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_model_without_cat.fit(X_rf_tr_without_cat,y_rf_tr_whithout_cat)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total nu

In [32]:
y_hat_rf_whithout_cat=rf_model_without_cat.predict(X_rf_val_withoout_cat)
acc=accuracy_score(y_rf_val_without_cat,y_hat_rf_whithout_cat)
print(f"accuracy RF without cat features {acc:.4f}")
print(classification_report(y_rf_val_without_cat,y_hat_rf_whithout_cat))

accuracy RF without cat features 0.5041
              precision    recall  f1-score   support

       False       0.50      0.62      0.55     41659
        True       0.51      0.39      0.44     41625

    accuracy                           0.50     83284
   macro avg       0.50      0.50      0.50     83284
weighted avg       0.50      0.50      0.50     83284



- without categorical code 50.41% vs 50.59% with them 

## iii) LightGBM

In [33]:
model_lgbm=lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
model_lgbm.fit(X_rf_tr_without_cat,y_rf_tr_whithout_cat)

y_hat_lgbm=model_lgbm.predict(X_rf_val_withoout_cat)
acc_lgbm=accuracy_score(y_rf_val_without_cat,y_hat_lgbm)
print(f"lightgbm accuracy : {acc_lgbm:.4f}")
print(classification_report(y_rf_val_without_cat,y_hat_lgbm))

lightgbm accuracy : 0.5057
              precision    recall  f1-score   support

       False       0.51      0.54      0.52     41659
        True       0.51      0.48      0.49     41625

    accuracy                           0.51     83284
   macro avg       0.51      0.51      0.51     83284
weighted avg       0.51      0.51      0.51     83284



- in Quant finance small edge can still matter because it's applied to thousands of stocks every day 

In [34]:
importances_lgbm=pd.Series(model_lgbm.feature_importances_,index=X_rf_tr_without_cat.columns)
importances_lgbm.sort_values(ascending=False).head(20)

RET_std_20    422
RET_13        387
RET_3         382
RET_12        378
RET_15        377
RET_2         375
RET_17        375
RET_7         375
RET_4         374
RET_16        374
RET_6         373
RET_9         368
RET_11        367
RET_8         366
RET_20        364
RET_14        364
VOLUME_1      357
RET_5         354
RET_19        351
VOLUME_13     347
dtype: int32

- only 1 featured engineered features is in top 20 !! 
- default metric in feature importance in lightgbm is split count ( ow many time a feature used to split across all trees )!= gain(how much each split improves the model) in random forest 

In [35]:
lgb_model_gain = lgb.LGBMClassifier(n_estimators=500, max_depth=8, random_state=42, n_jobs=-1, importance_type='gain')
lgb_model_gain.fit(X_rf_tr_without_cat, y_rf_tr_whithout_cat)
importances_gain = pd.Series(lgb_model_gain.feature_importances_, index=X_rf_tr_without_cat.columns)
importances_gain.sort_values(ascending=False).head(20)

RET_std_20    6809.002556
VOLUME_1      5183.240700
RET_7         4977.860965
RET_4         4896.478495
RET_3         4796.788365
RET_16        4614.741537
RET_9         4526.006487
RET_6         4490.039722
RET_2         4480.652322
RET_12        4441.624669
RET_13        4429.803174
RET_17        4420.815471
RET_11        4324.163972
RET_14        4236.441432
RET_20        4209.695557
RET_mean_5    4164.123064
RET_5         4162.866969
RET_8         4154.274498
RET_15        4085.223561
RET_10        4077.558808
dtype: float64

- RET_std_20 is 1st and volume 1 2nd  same top 2 as RF! 
- volume 1 rank supports building volume features in v2 

## as a next step we're going to score every model on all 5 data folds saving the results on a log

## 2 : phase 2 : 5 fold date CV all models 

In [62]:
print(FEATURE_VERSION, X_train_features.shape, X.shape)

v2 (418595, 97) (418595, 91)


In [63]:
def cv_score(model):
    scores=[]
    for tr_i,va_i in gkf.split(X,y,groups=dates):
        model.fit(X.iloc[tr_i],y.iloc[tr_i])
        scores.append(accuracy_score(y.iloc[va_i],model.predict(X.iloc[va_i])))
    return np.mean(scores),np.std(scores)

In [64]:
params=dict(n_estimators=300,learning_rate=0.02,num_leaves=15,min_child_samples=500,
              subsample=0.7,subsample_freq=1,colsample_bytree=0.5,reg_lambda=10,
              random_state=42,n_jobs=-1,verbose=-1)

In [65]:
models={
    "always False":DummyClassifier(strategy="most_frequent"),
    "logistic regression":LogisticRegression(max_iter=1000),
    "RF (benchmark)":RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42, n_jobs=-1),
    "LGBM default":lgb.LGBMClassifier(n_estimators=500, max_depth=8, random_state=42, n_jobs=-1, verbose=-1),
    "LGBM regularized":lgb.LGBMClassifier(**params),
}

In [66]:
rows=[]
for name,model in models.items():
    mean,std=cv_score(model)
    rows.append({"version":FEATURE_VERSION,"model":name,"acc_mean":round(mean,4),"acc_std":round(std,4)})
    print(f"{name:22s} {mean:.4f} +- {std:.4f}")

always False           0.5011 +- 0.0008
logistic regression    0.5063 +- 0.0027
RF (benchmark)         0.5160 +- 0.0064
LGBM default           0.5100 +- 0.0036
LGBM regularized       0.5147 +- 0.0062


- v2 ( 4 volume+ 2 peer features) cahnges each model by -0.07 to +0.14 points well inside the fold to fold spread 
- best v1(LGBM regularized 51.54%) & best v2 (RF, 51.6%)
- CC : on this data volume don't add mesurable signal beyond v1 

- 5 fold date CV, v1 features.
- our best is +0.23 points compred to the benchmark 51.31% ( BUT THE BENCHMARK IS CORED ON DIFFERENT DAYS SO IT'S INDICATIVE )
- RF AND REGULARIZED LGBM ARE TIED 
- Limiting LGBM (ie regularized ) stops it memorizing noise 

In [48]:
log_path=ROOT_DIR/"results.csv"
pd.DataFrame(rows).to_csv(log_path, mode="a",header=not log_path.exists(),index=False)

In [67]:
rf=RandomForestClassifier(n_estimators=500,max_depth=8,random_state=42,n_jobs=-1)
scores=[]
for tr_i, va_i in gkf.split(X,y,groups=dates):
    rf.fit(X.iloc[tr_i],y.iloc[tr_i])
    p=pd.Series(rf.predict_proba(X.iloc[va_i])[:, 1],index=X.index[va_i])
    rank_in_day=p.groupby(dates.iloc[va_i]).rank(pct=True)
    scores.append(accuracy_score(y.iloc[va_i], rank_in_day > 0.5))
print(f"RF v2, top half of day: {np.mean(scores):.4f} +- {np.std(scores):.4f}")

RF v2, top half of day: 0.5174 +- 0.0053


### load V2 test file & check it maches training

In [68]:
X_test_v2=pd.read_csv(DIR_DATA_PROCESSED/"x_test_features_v2.csv", index_col="ID")
X_test_b=X_test_v2.drop(columns=categorical_cols)
print(X.shape, X_test_b.shape)
print("same columns, same order:", list(X.columns) == list(X_test_b.columns))

(418595, 91) (198429, 91)
same columns, same order: True


### trian RF on all V2 training rows and predict the test set

In [69]:
rf_final=RandomForestClassifier(n_estimators=500,max_depth=8,random_state=42,n_jobs=-1)
rf_final.fit(X, y)
p_test_b=pd.Series(rf_final.predict_proba(X_test_b)[:, 1],index=X_test_b.index)
print(p_test_b.shape,p_test_b.describe())

(198429,) count    198429.000000
mean          0.498985
std           0.017037
min           0.397017
25%           0.488116
50%           0.500502
75%           0.511212
max           0.622330
dtype: float64


In [70]:
dates_test_b=X_test_v2["DATE"]
rank_b=p_test_b.groupby(dates_test_b).rank(pct=True)
pred_b=rank_b > 0.5
submission_b=pd.DataFrame({"ID": X_test_b.index,"RET": pred_b.values})
submission_b.to_csv(ROOT_DIR / "submission_B_rf_v2.csv", index=False)

ex=pd.read_csv(DIR_DATA_RAW/"test_rand.csv")
print(submission_b.shape, submission_b["RET"].mean())
print("same IDs, same order:",(ex["ID"].values==submission_b["ID"].values).all())
print("same columns:",list(ex.columns) == list(submission_b.columns),"| dtype:", submission_b["RET"].dtype)

(198429, 2) 0.5000680344102929
same IDs, same order: True
same columns: True | dtype: bool


In [71]:
a = pd.read_csv(ROOT_DIR / "submission.csv")
print("agreement with A:", (a["RET"].values == submission_b["RET"].values).mean())

agreement with A: 0.8549203997399574


In [72]:
cols = ["RET_1", "RET_mean_5", "RET_mean_20", "RET_std_5", "RET_std_20", "RET_REVERSAL", "VOLUME_1"]
check = pd.DataFrame({
    "train mean": X_train_features[cols].mean(),
    "test mean":  X_test_v2[cols].mean(),
    "train std":  X_train_features[cols].std(),
    "test std":   X_test_v2[cols].std(),
})
print(check.round(4))

              train mean  test mean  train std  test std
RET_1             0.0014     0.0004     0.0311    0.0300
RET_mean_5        0.0008     0.0009     0.0133    0.0130
RET_mean_20       0.0006     0.0006     0.0066    0.0064
RET_std_5         0.0230     0.0225     0.0215    0.0203
RET_std_20        0.0251     0.0256     0.0183    0.0187
RET_REVERSAL      0.0008    -0.0001     0.0302    0.0290
VOLUME_1         -0.1059    -0.1168     2.7939    4.7336
